# Entraîner des réseaux de neurones

Le réseau construit dans la partie précédente n’est pas très intelligent : il ne connaît rien de nos chiffres manuscrits. Les réseaux de neurones avec des activations non linéaires agissent comme des approximateurs universels de fonctions. Il existe une fonction qui associe votre entrée à la sortie, par exemple des images de chiffres manuscrits à des probabilités de classe. La force des réseaux de neurones est que nous pouvons les entraîner à approximer cette fonction — et pratiquement n’importe quelle fonction — avec suffisamment de données et de temps de calcul.

<img src="assets/function_approx.png" width=500px>

Au départ, le réseau est naïf : il ne connaît pas la fonction qui associe les entrées aux sorties. Nous l’entraînons en lui montrant des exemples de données réelles, puis en ajustant ses paramètres afin qu’il approxime cette fonction.

Pour trouver ces paramètres, nous devons savoir dans quelle mesure le réseau prédit mal les sorties réelles. Nous calculons pour cela une **fonction de perte** (aussi appelée coût), qui mesure l’erreur de prédiction. Par exemple, la perte quadratique moyenne est souvent utilisée pour les problèmes de régression et de classification binaire.

$$
\large \ell = \frac{1}{2n}\sum_i^n{\left(y_i - \hat{y}_i\right)^2}
$$

où $n$ est le nombre d’exemples d’entraînement, $y_i$ sont les vraies étiquettes et $\hat{y}_i$ les étiquettes prédites.

En minimisant cette perte par rapport aux paramètres du réseau, nous pouvons trouver des configurations où elle est minimale et où le réseau prédit correctement les étiquettes avec une grande précision. Nous trouvons ce minimum grâce à la **descente de gradient**. Le gradient est la pente de la fonction de perte et indique la direction de changement la plus rapide. Pour atteindre le minimum efficacement, nous suivons le gradient vers le bas, comme lorsque l’on descend une montagne par la pente la plus raide.

<img src='assets/gradient_descent.png' width=350px>

## Rétropropagation

Pour les réseaux à une seule couche, la descente de gradient est simple à implémenter. Elle est toutefois plus complexe pour les réseaux profonds à plusieurs couches comme celui que nous avons construit. Il a fallu environ 30 ans aux chercheurs pour comprendre comment entraîner ces réseaux multicouches.

L’entraînement des réseaux multicouches s’effectue par **rétropropagation**, qui n’est en réalité qu’une application de la règle de la chaîne du calcul différentiel. Il est plus facile de la comprendre en représentant un réseau à deux couches sous forme de graphe.

<img src='assets/backprop_diagram.png' width=550px>

Lors de la passe avant dans le réseau, les données et les opérations vont ici du bas vers le haut. Nous faisons passer l’entrée $x$ dans une transformation linéaire $L_1$ ayant pour poids $W_1$ et pour biais $b_1$. La sortie passe ensuite par l’opération sigmoïde $S$, puis par une autre transformation linéaire $L_2$. Enfin, nous calculons la perte $\ell$, qui mesure la qualité des prédictions du réseau. L’objectif est d’ajuster les poids et les biais afin de minimiser cette perte.

Pour entraîner les poids par descente de gradient, nous propageons le gradient de la perte en arrière dans le réseau. Chaque opération possède un gradient entre ses entrées et ses sorties. En faisant circuler les gradients vers l’arrière, nous multiplions le gradient entrant par celui de l’opération. Mathématiquement, il s’agit de calculer le gradient de la perte par rapport aux poids à l’aide de la règle de la chaîne.

$$
\large \frac{\partial \ell}{\partial W_1} = \frac{\partial L_1}{\partial W_1} \frac{\partial S}{\partial L_1} \frac{\partial L_2}{\partial S} \frac{\partial \ell}{\partial L_2}
$$

**Remarque :** certains détails relevant du calcul vectoriel sont volontairement simplifiés ici ; ils ne sont pas nécessaires pour comprendre le principe.

Nous mettons à jour les poids à l’aide de ce gradient et d’un taux d’apprentissage $\alpha$. 

$$
\large W^\prime_1 = W_1 - \alpha \frac{\partial \ell}{\partial W_1}
$$

Le taux d’apprentissage $\alpha$ est choisi pour que les mises à jour des poids soient suffisamment petites afin que la méthode itérative converge vers un minimum.

## Fonctions de perte dans PyTorch

Commençons par voir comment calculer la perte avec PyTorch. Par l’intermédiaire du module `nn`, PyTorch fournit des fonctions de perte telles que l’entropie croisée (`nn.CrossEntropyLoss`). La fonction de perte est généralement affectée à `criterion`. Comme indiqué dans la partie précédente, pour un problème de classification tel que MNIST, nous utilisons la fonction softmax afin de prédire les probabilités des classes. Avec une sortie softmax, il est préférable d’utiliser l’entropie croisée comme perte. Pour calculer la perte, définissez d’abord le critère, puis fournissez-lui la sortie du réseau et les bonnes étiquettes.

Un point très important : En consultant [la documentation de `nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/nn.html#torch.nn.CrossEntropyLoss) :

> Ce critère combine `nn.LogSoftmax()` et `nn.NLLLoss()` dans une seule classe.
>
> L’entrée doit contenir les scores de chaque classe.

Nous devons donc fournir à la perte la sortie brute du réseau, et non la sortie de la fonction softmax. Cette sortie brute est généralement appelée *logits* ou *scores*. Nous utilisons les logits, car softmax produit souvent des probabilités très proches de zéro ou de un, que les nombres à virgule flottante ne représentent pas précisément ([en savoir plus](https://docs.python.org/3/tutorial/floatingpoint.html)). Il est généralement préférable d’éviter les calculs avec des probabilités et d’utiliser plutôt des log-probabilités.

In [ ]:
# The MNIST datasets are hosted on yann.lecun.com that has moved under CloudFlare protection
# Run this script to enable the datasets download
# Reference: https://github.com/pytorch/vision/issues/1938

from six.moves import urllib
opener = urllib.request.build_opener()
opener.addheaders = [('User-agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms

# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,)),
                              ])
# Download and load the training data
trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

### Remarque
Si vous n’avez pas encore vu `nn.Sequential`, terminez d’abord la fin du notebook de la partie 2.

In [ ]:
# Build a feed-forward network
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10))

# Define the loss
criterion = nn.CrossEntropyLoss()

# Get our data
dataiter = iter(trainloader)

images, labels = next(dataiter)

# Flatten images
images = images.view(images.shape[0], -1)

# Forward pass, get our logits
logits = model(images)
# Calculate the loss with the logits and the labels
loss = criterion(logits, labels)

print(loss)

D’après mon expérience, il est plus pratique de construire le modèle avec une sortie log-softmax à l’aide de `nn.LogSoftmax` ou de `F.log_softmax` ([documentation](https://pytorch.org/docs/stable/nn.html#torch.nn.LogSoftmax)). Les probabilités réelles s’obtiennent ensuite en calculant l’exponentielle `torch.exp(output)`. Avec une sortie log-softmax, utilisez la perte de log-vraisemblance négative `nn.NLLLoss` ([documentation](https://pytorch.org/docs/stable/nn.html#torch.nn.NLLLoss)).

>**Exercice :** construisez un modèle qui renvoie le log-softmax en sortie et calculez la perte avec la log-vraisemblance négative. Pour `nn.LogSoftmax` et `F.log_softmax`, choisissez correctement l’argument nommé `dim`. Avec `dim=0`, softmax est calculé sur les lignes et chaque colonne somme à 1 ; avec `dim=1`, il est calculé sur les colonnes et chaque ligne somme à 1. Réfléchissez à la sortie souhaitée pour choisir `dim`.

In [ ]:
# TODO: Build a feed-forward network
model = 

# TODO: Define the loss
criterion = 

### Run this to check your work
# Get our data
dataiter = iter(trainloader)

images, labels = next(dataiter)

# Flatten images
images = images.view(images.shape[0], -1)

# Forward pass, get our logits
logits = model(images)
# Calculate the loss with the logits and the labels
loss = criterion(logits, labels)

print(loss)

## Autograd

Maintenant que nous savons calculer une perte, comment l’utiliser pour réaliser la rétropropagation ? Torch fournit le module `autograd`, qui calcule automatiquement les gradients des tenseurs. Nous pouvons l’utiliser pour calculer les gradients de tous les paramètres par rapport à la perte. Autograd conserve la trace des opérations effectuées sur les tenseurs, puis les parcourt en sens inverse en calculant les gradients. Pour que PyTorch suive les opérations et calcule les gradients d’un tenseur, définissez `requires_grad = True` lors de sa création ou à tout moment avec `x.requires_grad_(True)`.

Vous pouvez désactiver les gradients pour un bloc de code avec le contexte `torch.no_grad()` :
```python
x = torch.zeros(1, requires_grad=True)
>>> with torch.no_grad():
...     y = x * 2
>>> y.requires_grad
False
```

Vous pouvez aussi activer ou désactiver complètement les gradients avec `torch.set_grad_enabled(True|False)`.

Les gradients sont calculés par rapport à une variable `z` avec `z.backward()`. Cela effectue une passe arrière à travers les opérations ayant créé `z`.

In [ ]:
x = torch.randn(2,2, requires_grad=True)
print(x)

In [ ]:
y = x**2
print(y)

Ci-dessous, nous voyons l’opération qui a créé `y` : une opération de puissance `PowBackward0`.

In [ ]:
## grad_fn shows the function that generated this variable
print(y.grad_fn)

Le module autograd suit ces opérations et sait calculer le gradient de chacune d’elles. Il peut ainsi calculer les gradients d’une chaîne d’opérations par rapport à n’importe quel tenseur. Réduisons le tenseur `y` à une valeur scalaire : la moyenne.

In [ ]:
z = y.mean()
print(z)

Vous pouvez vérifier les gradients de `x` et de `y`, mais ils sont actuellement vides.

In [ ]:
print(x.grad)

Pour calculer les gradients, exécutez la méthode `.backward` sur une variable, par exemple `z`. Cela calcule le gradient de `z` par rapport à `x`.

$$
\frac{\partial z}{\partial x} = \frac{\partial}{\partial x}\left[\frac{1}{n}\sum_i^n x_i^2\right] = \frac{x}{2}
$$

In [ ]:
z.backward()
print(x.grad)
print(x/2)

Ces calculs de gradient sont particulièrement utiles pour les réseaux de neurones. Pour l’entraînement, nous avons besoin des gradients du coût par rapport aux poids. Avec PyTorch, nous faisons passer les données vers l’avant dans le réseau pour calculer la perte, puis vers l’arrière pour calculer les gradients. Une fois les gradients obtenus, nous pouvons effectuer une étape de descente de gradient. 

## Perte et Autograd ensemble

Lorsque nous créons un réseau avec PyTorch, tous ses paramètres sont initialisés avec `requires_grad = True`. Ainsi, lorsque nous calculons la perte et appelons `loss.backward()`, les gradients des paramètres sont calculés. Ces gradients servent à mettre les poids à jour par descente de gradient. Vous trouverez ci-dessous un exemple de leur calcul lors d’une passe arrière.

In [ ]:
# Build a feed-forward network
model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10),
                      nn.LogSoftmax(dim=1))

criterion = nn.NLLLoss()
dataiter = iter(trainloader)
images, labels = next(dataiter)
images = images.view(images.shape[0], -1)

logits = model(images)
loss = criterion(logits, labels)

In [ ]:
print('Before backward pass: \n', model[0].weight.grad)

loss.backward()

print('After backward pass: \n', model[0].weight.grad)

## Entraîner le réseau !

Il nous manque un dernier élément pour commencer l’entraînement : un optimiseur, qui mettra à jour les poids à l’aide des gradients. PyTorch les fournit dans son [package `optim`](https://pytorch.org/docs/stable/optim.html). Par exemple, nous pouvons utiliser la descente de gradient stochastique avec `optim.SGD`. Voici comment définir un optimiseur.

In [ ]:
from torch import optim

# Optimizers require the parameters to optimize and a learning rate
optimizer = optim.SGD(model.parameters(), lr=0.01)

Nous savons maintenant utiliser tous les éléments séparément ; voyons comment ils fonctionnent ensemble. Considérons une seule étape d’apprentissage avant de parcourir toutes les données. Le processus général dans PyTorch est le suivant :

* Effectuer une passe avant dans le réseau 
* Utiliser la sortie du réseau pour calculer la perte
* Effectuer une passe arrière avec `loss.backward()` pour calculer les gradients
* Faire une étape avec l’optimiseur pour mettre à jour les poids

Ci-dessous, nous allons parcourir une étape d’entraînement et afficher les poids et les gradients pour observer leur évolution. Notez la ligne `optimizer.zero_grad()`. Lorsque plusieurs passes arrière sont effectuées avec les mêmes paramètres, les gradients s’accumulent. Il faut donc les remettre à zéro à chaque passe d’entraînement, sous peine de conserver les gradients des lots précédents.

In [ ]:
print('Initial weights - ', model[0].weight)

dataiter = iter(trainloader)
images, labels = next(dataiter)
images.resize_(64, 784)

# Clear the gradients, do this because gradients are accumulated
optimizer.zero_grad()

# Forward pass, then backward pass, then update weights
output = model(images)
loss = criterion(output, labels)
loss.backward()
print('Gradient -', model[0].weight.grad)

In [ ]:
# Take an update step and view the new weights
optimizer.step()
print('Updated weights - ', model[0].weight)

### Entraînement complet

Plaçons maintenant cet algorithme dans une boucle afin de parcourir toutes les images. Une passe complète sur le jeu de données est appelée une *époque*. Nous allons donc parcourir `trainloader` pour récupérer les lots d’entraînement. Pour chaque lot, nous calculerons la perte, effectuerons une passe arrière et mettrons les poids à jour.

>**Exercice :** implémentez la passe d’entraînement de notre réseau. Si votre implémentation est correcte, la perte d’entraînement doit diminuer à chaque époque.

In [ ]:
## Your solution here

model = nn.Sequential(nn.Linear(784, 128),
                      nn.ReLU(),
                      nn.Linear(128, 64),
                      nn.ReLU(),
                      nn.Linear(64, 10),
                      nn.LogSoftmax(dim=1))

criterion = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.003)

epochs = 5
for e in range(epochs):
    running_loss = 0
    for images, labels in trainloader:
        # Flatten MNIST images into a 784 long vector
        images = images.view(images.shape[0], -1)
    
        # TODO: Training pass
        
        loss = 
        
        running_loss += loss.item()
    else:
        print(f"Training loss: {running_loss/len(trainloader)}")

Une fois le réseau entraîné, nous pouvons examiner ses prédictions.

In [ ]:
%matplotlib inline
import helper

dataiter = iter(trainloader)
images, labels = next(dataiter)

img = images[0].view(1, 784)
# Turn off gradients to speed up this part
with torch.no_grad():
    logps = model(img)

# Output of the network are log-probabilities, need to take exponential for probabilities
ps = torch.exp(logps)
helper.view_classify(img.view(1, 28, 28), ps)

Notre réseau est maintenant performant : il peut prédire avec précision les chiffres de nos images. Dans la suite, vous écrirez le code pour entraîner un réseau de neurones sur un jeu de données plus complexe.